# Source 3 (File) — airports.csv

This is a reference of airports (name, codes, coordinates, country) used for:
- **Busiest airports (Q3)** — spatial-join low climbing/descending aircraft to their nearest airport (needs `latitude_deg`/`longitude_deg`).
- **Labels & geography** — human-readable airport names + country/continent for the maps and routes.

- File: `https://raw.githubusercontent.com/davidmegginson/ourairports-data/refs/heads/main/airports.csv`

## "Problem" with the dataset/the processing:
The `continent` column uses `NA` for North America, so `read_csv` silently turns 39,622 North-American airports into `NaN`.

Fix: read with `keep_default_na=False` (and only treat empty strings as missing). The cell below shows the difference.

This would not have been a big problem for this project, as we focus on Europe. But it's just generally prettier if the problem's actually fixed.

In [3]:
import pandas as pd

URL = "https://raw.githubusercontent.com/davidmegginson/ourairports-data/refs/heads/main/airports.csv"

naive = pd.read_csv(URL)
print("continent counts — NAIVE read ('NA' becomes NaN):")
print(naive["continent"].value_counts(dropna=False))

df = pd.read_csv(URL, keep_default_na=False, na_values=[""])
print("\ncontinent counts — CORRECT read (keep_default_na=False):")
print(df["continent"].value_counts(dropna=False))

continent counts — NAIVE read ('NA' becomes NaN):
continent
NaN    39622
AS     12729
EU     12506
SA     12272
AF      4231
OC      4221
AN        47
Name: count, dtype: int64

continent counts — CORRECT read (keep_default_na=False):
continent
NA    39622
AS    12729
EU    12506
SA    12272
AF     4231
OC     4221
AN       47
Name: count, dtype: int64


## Structure

In [5]:
df.shape

(85628, 19)

Ober 85000 airports, 19 columns. The ones we care about:

| column | meaning | use for us |
|--------|---------|------------|
| `type` | airport size/kind | keep `large_airport` / `medium_airport` |
| `name` | airport name | labels |
| `latitude_deg` / `longitude_deg` | position | spatial join (Q3) |
| `continent` | continent code | filter to `EU` |
| `iso_country` | country (ISO) | geography |
| `scheduled_service` | has scheduled flights? | keep `yes` (commercial) |
| `icao_code` | 4-letter ICAO | join key to flights / routes |
| `iata_code` | 3-letter IATA | labels / routes |

In [7]:
print("\ntypes:")
print(df["type"].value_counts())
print("\nscheduled_service:")
print(df["scheduled_service"].value_counts())


types:
type
small_airport     42658
heliport          23089
closed            13279
medium_airport     4097
seaplane_base      1266
large_airport      1178
balloonport          61
Name: count, dtype: int64

scheduled_service:
scheduled_service
no     81225
yes     4403
Name: count, dtype: int64


## Filter to European commercial airports
For our analysis we want Europe + scheduled service + real airports (large/medium). That is the set we will spatial-join live flights against.

In [8]:
eu = df[
    (df["continent"] == "EU")
    & (df["scheduled_service"] == "yes")
    & (df["type"].isin(["large_airport", "medium_airport"]))
].copy()

print("European commercial airports:", len(eu))
print("  large_airport :", (eu["type"] == "large_airport").sum())
print("  medium_airport:", (eu["type"] == "medium_airport").sum())
print()
# data quality of the fields we depend on
for col in ["icao_code", "iata_code", "latitude_deg", "longitude_deg"]:
    missing = eu[col].isna().sum() + (eu[col] == "").sum() if eu[col].dtype == object else eu[col].isna().sum()
    print(f"missing {col}: {missing}")

European commercial airports: 528
  large_airport : 256
  medium_airport: 272

missing icao_code: 0
missing iata_code: 2
missing latitude_deg: 0
missing longitude_deg: 0


In [9]:
# Sample of the big European hubs we expect to dominate Q3
cols = ["icao_code", "iata_code", "name", "iso_country", "latitude_deg", "longitude_deg"]
eu[eu["type"] == "large_airport"][cols].head(10)

,icao_code,iata_code,name,iso_country,latitude_deg,longitude_deg
12992,BIAR,AEY,Akureyri International Airport,IS,65.656573,-18.072018
13029,BIKF,KEF,Keflavik International Airport,IS,63.985001,-22.605600
13072,BKPR,PRN,Priština Adem Jashari International Airport,XK,42.572800,21.035801
23160,EBBR,BRU,Brussels Airport,BE,50.901402,4.484440
23166,EBCI,CRL,Brussels South Charleroi Airport,BE,50.461963,4.459562
23235,EBOS,OST,Ostend-Bruges International Airport,BE,51.199800,2.874673
23417,EDDB,BER,Berlin Brandenburg Airport,DE,52.361738,13.502341
23418,EDDC,DRS,Dresden Airport,DE,51.134123,13.767831
23419,EDDE,ERF,Erfurt-Weimar Airport,DE,50.978281,10.960713
23420,EDDF,FRA,Frankfurt Main Airport,DE,50.026706,8.558350
